<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# Time Series with Pandas
---
<small>© COPYRIGHT 2023 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/ </small>


<!-- TIP: Use HTML comment boxes (<details><summary>Tip</summary>...</details>) throughout this notebook to inspect code hints and solutions for time series forecasting with Pandas and AutoGluon. -->


In [ ]:
# Load core data manipulation and analysis libraries
import pandas as pd
import numpy as np
# Additional statsmodels or time series packages (e.g. pmdarima, statsmodels)


Load in your data in the usual way (such as with `pd.read_csv('your_file.csv')`).

<!-- TIP: Use pd.read_csv("AucklandTemperatureData.csv") to load the dataset into a pandas DataFrame -->
<details>
<summary>💡 <b>Tip: Loading CSV Data</b></summary>
<!-- HINT: Pass filename to pd.read_csv() -->
<code>auckland = pd.read_csv("AucklandTemperatureData.csv")</code>
</details>


In [ ]:
# Read the Auckland temperature dataset into a DataFrame and preview the first 5 rows
auckland = pd.read_csv("AucklandTemperatureData.csv")
auckland.head()


In [ ]:
# Filter dataset slice to remove extreme boundaries / missing tail observations
auckland = auckland[-5000:-20]


In [ ]:
auckland

In [ ]:
# Create exploratory line plot of the Auckland temperature series
auckland.plot()


In [ ]:

# Create the DataFrame column names for the Auckland temperature dataset
auckland.columns

In [ ]:
# Convert index back to a standard column if datetime was set as index
# AutoGluon requires explicit timestamp and item_id columns
df = auckland.reset_index()
df.head()


In [ ]:
# Import AutoGluon TimeSeries components for automated model training and forecasting
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


In [ ]:
# Ensure index is reset to provide a clean timestamp column for AutoGluon
# Hint: df = auckland.reset_index()
df = auckland.reset_index()


In [ ]:
# Create unique item identifier column required by AutoGluon (single series = "auckland")
df["item_id"] = "auckland"


In [ ]:
# Define forecast horizon: number of future time steps to predict (e.g. 12 timesteps)
prediction_length = 12


In [ ]:
# Convert pandas DataFrame into AutoGluon TimeSeriesDataFrame
# Hint: Specify id_column and timestamp_column
train_data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column="item_id",
    timestamp_column=df.columns[0],
)


In [ ]:
# Slice dataset into train (excluding test period) and test (last prediction_length steps)
train = train_data.slice_by_timestep(None, -prediction_length)
test = train_data.slice_by_timestep(-prediction_length, None)


In [ ]:
# Initialize TimeSeriesPredictor specifying target variable and evaluation metric (MASE)
predictor = TimeSeriesPredictor(
    prediction_length=12,
    target="temperature",
    eval_metric="MASE",
)


In [ ]:
# Train automated time series forecasting models within a 60-second time budget
predictor.fit(train, time_limit=60)


In [ ]:
# Evaluate leaderboard ranking all fitted models based on validation performance
leaderboard = predictor.leaderboard()
leaderboard


In [ ]:
# Generate in-sample forecast predictions on the training data
forecast = predictor.predict(train)


In [ ]:
test

In [ ]:
# Plot ground truth test data against model forecasts for the item "auckland"
predictor.plot(
    test,
    forecast,
    item_ids=["auckland"]
)


In [ ]:
# Generate test set predictions for the top-performing model and baseline SeasonalNaive model
best_forecast = predictor.predict(test)
naive_forecast = predictor.predict(test, model="SeasonalNaive")


In [ ]:
# Compare visualizations of actual test values vs Best Model and Seasonal Naive baseline
predictor.plot(test, best_forecast, item_ids=["auckland"])
predictor.plot(test, naive_forecast, item_ids=["auckland"])


In [ ]:
# Custom Matplotlib plotting of actual test values vs best AutoGluon forecast
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(actual.index, actual.values, marker="o", label="Actual test values")
plt.plot(actual.index, best.values, marker="o", label="Forecast")

plt.title("Actual vs forecast over the same test period")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Detailed multi-model comparison plot (Actual vs Best Model vs Naive Baseline)
import matplotlib.pyplot as plt

item_id = "auckland"
target = "temperature"   # target column

actual = test.loc[item_id][target]
best = best_forecast.loc[item_id]["mean"]
naive = naive_forecast.loc[item_id]["mean"]

# Align indices for plotting
best.index = actual.index
naive.index = actual.index

plt.figure(figsize=(12, 5))

plt.plot(actual.index, actual.values, marker="o", linewidth=2, label="Actual")
plt.plot(actual.index, best.values, marker="o", linewidth=2, label="Best Model")
plt.plot(actual.index, naive.values, marker="o", linestyle="--", linewidth=2, label="Naive")

plt.title("Actual vs Forecasts (Test Period)")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Compute absolute errors for Best Model and Naive Baseline
import pandas as pd

comparison = pd.DataFrame({
    "actual": actual.values,
    "best": best.values,
    "naive": naive.values,
}, index=actual.index)

comparison["best_abs_error"] = (comparison["actual"] - comparison["best"]).abs()
comparison["naive_abs_error"] = (comparison["actual"] - comparison["naive"]).abs()

comparison


In [ ]:
# Calculate Mean Absolute Error (MAE) for Best Model vs Naive Baseline
comparison[["best_abs_error", "naive_abs_error"]].mean()


In [ ]:
predictor.eval_metric

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(actual.index, actual.values, marker="o", label="Actual")
plt.plot(best.index, best.values, marker="o", label="Best model")
plt.plot(naive.index, naive.values, marker="o", linestyle="--", label="Naive")

plt.title("Actual vs Forecasts: Test Period")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "actual": actual,
    "best": best,
    "naive": naive,
})

comparison["best_abs_error"] = (comparison["actual"] - comparison["best"]).abs()
comparison["naive_abs_error"] = (comparison["actual"] - comparison["naive"]).abs()

comparison